In [1]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [2]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [3]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [7]:
from google import genai
from dotenv import load_dotenv
from evaluation_utils import calc_price,calc_total_price,llm_structured_retry
load_dotenv()
gemini_client = genai.Client()

In [12]:
import pandas as pd
import random

df_answers = pd.read_csv("../data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

random.seed(42)
answers = random.sample(answers, min(150, len(answers)))
print(f"Judging {len(answers)} answers")

Judging 150 answers


In [14]:
rec = answers[0]
rec


{'question': 'Where can I find a list of free options instead of paying for OpenAI?',
 'answer_llm': 'You can find a list of free options by checking the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives) or by visiting [Awesome LLMs](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md) in the course repo.',
 'answer_orig': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives).",
 'document': '85384a18e5'}

In [9]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
Is it too late to enroll since I just found out about the program?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list.


In [16]:
eval_result, usage = llm_structured_retry(
    gemini_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning="The original answer implies enrollment is technically past the deadline, but submission is still possible for certification. The AI answer clarifies that registration isn't strictly enforced for participation/submission, which aligns with the practical reality of the original answer's intent.", score='good')

In [17]:
calc_price(usage)

{'input_cost': 3.5300000000000004e-05,
 'output_cost': 2.68e-05,
 'total_cost': 6.21e-05}

In [20]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gemini-3.1-flash-lite"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        gemini_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [21]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer provides the correct link mentioned in the original answer and accurately conveys the information that free alternatives are available in the course repository.', score='good')

In [22]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [ ]:
import time
import os
from tqdm import tqdm

out_path = "../data/rag-evaluations-new.csv"
evaluations = []
usages = []


done_questions = set()
if os.path.exists(out_path):
    done_questions = set(pd.read_csv(out_path)["question"])

write_header = not os.path.exists(out_path)

for rec in tqdm(answers):
    if rec["question"] in done_questions:
        continue

    evaluation, usage = judge_record(rec)
    evaluations.append(evaluation)
    usages.append(usage)

    pd.DataFrame([evaluation]).to_csv(out_path, mode="a", header=write_header, index=False)
    write_header = False

    time.sleep(4.5)

df_eval = pd.read_csv(out_path) 
print(f"Total evaluations: {len(df_eval)}")
print(f"Total cost: ${calc_total_price(usages):.2f}")

100%|██████████| 150/150 [13:31<00:00,  5.41s/it]

Total evaluations: 150
Total cost: $0.01


In [24]:
calc_total_price(usages)

0.011239699999999995

In [25]:
df_eval = pd.DataFrame(evaluations)

In [26]:
df_eval.head()

,question,document,score,reasoning
0,Where can I find a list of free options instea...,85384a18e5,good,The AI answer provides the correct link mentio...
1,Can I still earn my certificate if I decide to...,69d122f12e,good,The AI answer accurately captures the core req...
2,Can you explain why keeping the original conte...,69430a79a8,bad,The AI assistant failed to provide any informa...
3,Why am I getting a 402 error from OpenRouter w...,cfb07a27d5,good,The AI answer correctly identifies the core ca...
4,Do I need to change how I define the tools sch...,9e7b3f0c25,bad,While the AI answer correctly identifies that ...


In [27]:
df_eval.score.value_counts()

score
good    105
bad      45
Name: count, dtype: int64

In [28]:
df_eval.score.value_counts(normalize=True)

score
good    0.7
bad     0.3
Name: proportion, dtype: float64

In [30]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
2,Can you explain why keeping the original conte...,69430a79a8,bad,The AI assistant failed to provide any informa...
4,Do I need to change how I define the tools sch...,9e7b3f0c25,bad,While the AI answer correctly identifies that ...
9,How can I stop the columns from showing those ...,641aedbbf5,bad,"The AI assistant failed to provide the answer,..."
10,What specifically do I need to finish in order...,9f689c185f,bad,The AI answer is incomplete because it incorre...
14,What is the difference between OpenAIChatCompl...,0224d980b3,bad,The AI assistant claimed it did not know the a...
